In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# File path for the merged dataset
file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_with_persons.csv"

# Load the dataset into a DataFrame
df = pd.read_csv(file_path)

# Display basic information about the dataset
print("Dataset Shape:", df.shape)         # Dimensions of the dataset
print("\nDataFrame Info:")
print(df.info())                          # Data types and non-null counts

print("\nFirst 5 Rows:")
print(df.head())                          # First few rows to see sample data

print("\nSummary Statistics:")
print(df.describe(include='all'))         # Statistical summary (for both numeric and categorical columns)

print("\nMissing Values per Column:")
print(df.isnull().sum())                  # Count of missing values per column


Dataset Shape: (5563805, 25)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5563805 entries, 0 to 5563804
Data columns (total 25 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Timestamp          object 
 1   Accelerometer X    float64
 2   Accelerometer Y    float64
 3   Accelerometer Z    float64
 4   Gyroscope X        float64
 5   Gyroscope Y        float64
 6   Gyroscope Z        float64
 7   Magnetometer X     float64
 8   Magnetometer Y     float64
 9   Magnetometer Z     float64
 10  Rotation Vector X  float64
 11  Rotation Vector Y  float64
 12  Rotation Vector Z  float64
 13  Tilt Detector X    float64
 14  Tilt Detector Y    float64
 15  Tilt Detector Z    float64
 16  Auto-rotation X    float64
 17  Auto-rotation Y    float64
 18  Auto-rotation Z    float64
 19  Motion X           float64
 20  Motion Y           float64
 21  Motion Z           float64
 22  Last Touch X       float64
 23  Last Touch Y       float64
 24  Pers

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler

# --- Step 1: Load Data ---
file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_with_persons.csv"
df = pd.read_csv(file_path)
print("Initial dataset shape:", df.shape)

# --- Step 2: Convert Timestamp to datetime ---
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# --- Step 3: Sort Data by Timestamp ---
df.sort_values('Timestamp', inplace=True)

# --- Step 4: Remove Duplicate Rows ---
df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)

# --- Step 5: Remove Outliers ---
# Define sensor columns (all columns except 'Timestamp' and 'Person')
sensor_columns = df.columns[1:24]  # columns 1 to 23 are sensor readings
# Compute absolute Z-scores for sensor data
z_scores = np.abs(zscore(df[sensor_columns]))
# Create a mask to filter rows where all sensor values have Z-score < 3
mask = (z_scores < 3).all(axis=1)
df_clean = df[mask].copy()
print("After outlier removal:", df_clean.shape)

# --- Step 6: Timestamp Bucketing with Person Grouping ---
# Floor the Timestamp to the nearest 100ms to create uniform time intervals
df_clean['Timestamp_bucket'] = df_clean['Timestamp'].dt.floor('10ms')

# Group by both Person and the floored timestamp, so each person's data is kept separate
agg_dict = {col: 'mean' for col in sensor_columns}
# For 'Timestamp', we'll take the first timestamp in the bucket; Person is preserved by grouping
df_grouped = df_clean.groupby(['Person', 'Timestamp_bucket']).agg(agg_dict).reset_index()
# Rename the bucket column back to 'Timestamp'
df_grouped.rename(columns={'Timestamp_bucket': 'Timestamp'}, inplace=True)
print("After timestamp bucketing by Person:", df_grouped.shape)

# --- Step 7: Normalize Sensor Data ---
scaler = MinMaxScaler()
df_grouped[sensor_columns] = scaler.fit_transform(df_grouped[sensor_columns])

# --- Step 8: Save the Preprocessed Data ---
output_file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_preprocessed.csv"
df_grouped.to_csv(output_file_path, index=False)
print("Preprocessing complete. Final preprocessed data saved to:", output_file_path)

Initial dataset shape: (5563805, 25)
After removing duplicates: (5563805, 25)
After outlier removal: (4944276, 25)
After timestamp bucketing by Person: (1290855, 25)
Preprocessing complete. Final preprocessed data saved to: /content/drive/MyDrive/Major Project/Combined Datas/merged_data_preprocessed.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler

# --- Step 1: Load Preprocessed Data in Chunks ---
file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_preprocessed.csv"
chunk_size = 100000  # Process 100,000 rows at a time

# --- Step 2: Open Output File for Incremental Writing ---
output_file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv"
is_first_chunk = True  # Track first chunk for writing headers

# --- Step 3: Process Data in Chunks ---
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    print("Processing new chunk...")

    # Ensure Timestamp column is in datetime format
    chunk.columns = chunk.columns.str.strip()
    if 'Timestamp' not in chunk.columns:
        print("⚠️ No 'Timestamp' column found. Skipping chunk.")
        continue

    chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
    chunk.dropna(subset=['Timestamp'], inplace=True)
    chunk.set_index('Timestamp', inplace=True)

    # Define sensor columns
    sensor_columns = [col for col in chunk.columns if col not in ['Person']]
    if not sensor_columns:
        print("⚠️ No sensor columns found in this chunk. Skipping.")
        continue

    # Compute rolling features (optimized)
    rolling_features = chunk.groupby('Person')[sensor_columns].rolling('20ms', min_periods=1).agg(['mean', 'std']).reset_index()

    if rolling_features.empty:
        print("⚠️ No valid rolling features computed. Skipping this chunk.")
        continue

    # Flatten MultiIndex columns correctly
    rolling_features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in rolling_features.columns]

    # Verify which columns exist before scaling
    available_columns = [col for col in rolling_features.columns if any(sensor in col for sensor in sensor_columns)]

    if available_columns:
        # Normalize only existing sensor columns
        scaler = StandardScaler()
        rolling_features[available_columns] = scaler.fit_transform(rolling_features[available_columns])
    else:
        print("⚠️ No valid columns available for normalization. Skipping this chunk.")

    # Save chunk to file (incremental write) in CSV format
    rolling_features.to_csv(output_file_path, mode='a', header=is_first_chunk, index=False)
    is_first_chunk = False  # Only write header for the first chunk

print("✅ Optimized Feature Engineering for LSTM & GRU Complete. Data saved to:", output_file_path)


Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
Processing new chunk...
✅ Optimized Feature Engineering for LSTM & GRU Complete. Data saved to: /content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv


In [ ]:
import pandas as pd

# Use chunking to avoid loading the entire file at once (if needed)
df_iter = pd.read_csv("/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv", nrows=5)
print("Columns in CSV:", df_iter.columns.tolist())


Columns in CSV: ['Person_', 'Timestamp_', 'Accelerometer X_mean', 'Accelerometer X_std', 'Accelerometer X_min', 'Accelerometer X_max', 'Accelerometer Y_mean', 'Accelerometer Y_std', 'Accelerometer Y_min', 'Accelerometer Y_max', 'Accelerometer Z_mean', 'Accelerometer Z_std', 'Accelerometer Z_min', 'Accelerometer Z_max', 'Gyroscope X_mean', 'Gyroscope X_std', 'Gyroscope X_min', 'Gyroscope X_max', 'Gyroscope Y_mean', 'Gyroscope Y_std', 'Gyroscope Y_min', 'Gyroscope Y_max', 'Gyroscope Z_mean', 'Gyroscope Z_std', 'Gyroscope Z_min', 'Gyroscope Z_max', 'Magnetometer X_mean', 'Magnetometer X_std', 'Magnetometer X_min', 'Magnetometer X_max', 'Magnetometer Y_mean', 'Magnetometer Y_std', 'Magnetometer Y_min', 'Magnetometer Y_max', 'Magnetometer Z_mean', 'Magnetometer Z_std', 'Magnetometer Z_min', 'Magnetometer Z_max', 'Rotation Vector X_mean', 'Rotation Vector X_std', 'Rotation Vector X_min', 'Rotation Vector X_max', 'Rotation Vector Y_mean', 'Rotation Vector Y_std', 'Rotation Vector Y_min', 'Rot

In [ ]:
import pandas as pd

unique_persons = set()
chunksize = 100000  # adjust based on available memory

for chunk in pd.read_csv("/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv", chunksize=chunksize):
    # Use the correct column name (e.g., "Person_" instead of "Person")
    unique_persons.update(chunk["Person_"].astype(str).unique())

print("Unique Person values in preprocessed CSV:", list(unique_persons))


<ipython-input-3-0b3a578482b2>:6: DtypeWarning: Columns (0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv", chunksize=chunksize):
<ipython-input-3-0b3a578482b2>:6: DtypeWarning: Columns (0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv", chunksize=chunksize):


Unique Person values in preprocessed CSV: ['20', '11', '15', '12', '7', '18', '14', '24', '3', '9', '5', 'Person_', '8', '16', '23', '17', '10', '13', '1', '6', '21', '4', '19', '25', '22', '2']


In [ ]:
import pandas as pd

unique_persons = set()
chunksize = 100000  # or another appropriate size

for chunk in pd.read_csv("/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv", chunksize=chunksize, low_memory=False):
    # Ensure the header row isn't included as data
    chunk = chunk[chunk["Person_"] != "Person_"]
    unique_persons.update(chunk["Person_"].astype(str).unique())

print("Unique Person values in preprocessed CSV:", list(unique_persons))


Unique Person values in preprocessed CSV: ['20', '11', '15', '12', '7', '18', '14', '24', '3', '9', '5', '8', '16', '23', '17', '10', '13', '1', '6', '21', '4', '19', '25', '22', '2']


In [ ]:
import numpy as np

# Load one appended array from y_train.npy to inspect its content
with open("/content/drive/MyDrive/Major Project/y_train.npy", 'rb') as f:
    y_sample = np.load(f, allow_pickle=True)
print("Unique labels in sample:", np.unique(y_sample.astype(str)))


Unique labels in sample: ['1']


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# 1. Load the Raw CSV
# -----------------------------
raw_file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_with_persons.csv"
df = pd.read_csv(raw_file_path, low_memory=False)  # low_memory=False to help with mixed types

# Ensure the "Person" column is treated as a string
if 'Person' in df.columns:
    df['Person'] = df['Person'].astype(str)
else:
    # If the column is named differently (e.g., "Person_"), rename it to "Person"
    df.rename(columns={"Person_": "Person"}, inplace=True)
    df['Person'] = df['Person'].astype(str)

# -----------------------------
# 2. Convert Timestamps and Clean Data
# -----------------------------
# Convert the Timestamp column to datetime; adjust column name if needed
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df.dropna(subset=['Timestamp'], inplace=True)
df.sort_values('Timestamp', inplace=True)
df.drop_duplicates(inplace=True)

# -----------------------------
# 3. Remove Outliers from Sensor Data
# -----------------------------
# Define sensor columns: assume these are all columns except "Person" and "Timestamp"
sensor_columns = [col for col in df.columns if col not in ['Person', 'Timestamp']]
# Compute z-scores for sensor columns
z_scores = np.abs(zscore(df[sensor_columns]))
# Create a mask to keep rows where every sensor value has a z-score less than 3
mask = (z_scores < 3).all(axis=1)
df_clean = df[mask].copy()

# -----------------------------
# 4. Timestamp Bucketing and Grouping
# -----------------------------
# Floor the Timestamp to the nearest 10 milliseconds to create uniform time intervals
df_clean['Timestamp_bucket'] = df_clean['Timestamp'].dt.floor('10ms')

# Define an aggregation dictionary:
# For sensor columns, take the mean. For the Person and Timestamp columns, use 'first' to preserve the original values.
agg_dict = {col: 'mean' for col in sensor_columns}
agg_dict['Person'] = 'first'
agg_dict['Timestamp'] = 'first'

# Group by Person and the floored timestamp; reset the index afterward
df_grouped = df_clean.groupby(['Person', 'Timestamp_bucket'], as_index=False).agg(agg_dict)

# Rename the bucket column back to "Timestamp"
df_grouped.rename(columns={'Timestamp_bucket': 'Timestamp'}, inplace=True)

# -----------------------------
# 5. Normalize Sensor Data
# -----------------------------
scaler = MinMaxScaler()
df_grouped[sensor_columns] = scaler.fit_transform(df_grouped[sensor_columns])

# -----------------------------
# 6. Save the Preprocessed Data
# -----------------------------
preprocessed_file_path = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv"
df_grouped.to_csv(preprocessed_file_path, index=False)

print("Preprocessing complete. Unique Person values in preprocessed CSV:", df_grouped["Person"].unique())


Preprocessing complete. Unique Person values in preprocessed CSV: ['1' '10' '11' '12' '13' '14' '15' '16' '17' '18' '19' '2' '20' '21' '22'
 '23' '24' '25' '3' '4' '5' '6' '7' '8' '9']


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# -----------------------------
# PARAMETERS
# -----------------------------
csv_file = "/content/drive/MyDrive/Major Project/Combined Datas/merged_data_lstm_glu_features.csv"
chunksize = 100000      # Number of rows per chunk
seq_len = 50            # Sliding window length (time steps per sequence)
batch_size = 64
shuffle_buffer_size = 10000  # Adjust to a value that can mix your data well

# -----------------------------
# STEP 1. Determine Sensor and Target Columns
# -----------------------------
# Load a small sample to get the column names.
df_sample = pd.read_csv(csv_file, nrows=10, low_memory=False)
all_cols = df_sample.columns.tolist()

# Assume the target column is named "Person" (or "Person_" then rename accordingly)
# Here we check for a column that looks like "Person"
target_col = None
for col in all_cols:
    if col.strip().lower().startswith("person"):
        target_col = col.strip()
        break
if target_col is None:
    raise ValueError("Could not find a column starting with 'Person'.")

# For sensor columns, exclude any column that contains "timestamp" or "person" (case-insensitive)
sensor_cols = [col for col in all_cols if "timestamp" not in col.lower() and "person" not in col.lower()]
print("Sensor columns:", sensor_cols)
print("Target column:", target_col)

# -----------------------------
# STEP 2. Create a Generator for Sequence Creation
# -----------------------------
def sequence_generator(csv_file, sensor_cols, target_col, seq_len, chunksize):
    """
    Reads the CSV file in chunks and yields sliding-window sequences.
    It uses a 'tail' to ensure continuity across chunks.
    """
    tail = None
    for chunk in pd.read_csv(csv_file, chunksize=chunksize, low_memory=False):
        # Ensure target column is string
        if target_col in chunk.columns:
            chunk[target_col] = chunk[target_col].astype(str)
        # Prepend tail from previous chunk if it exists.
        if tail is not None:
            chunk = pd.concat([tail, chunk], ignore_index=True)
        # Convert sensor columns to numeric (forcing errors to NaN)
        for col in sensor_cols:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce')
        # Drop rows with NaN in sensor columns
        chunk.dropna(subset=sensor_cols, inplace=True)
        n = len(chunk)
        if n <= seq_len:
            tail = chunk
            continue
        for i in range(n - seq_len):
            X_seq = chunk[sensor_cols].iloc[i:i+seq_len].values.astype(np.float32)
            y_label = chunk[target_col].iloc[i+seq_len]
            yield X_seq, y_label
        tail = chunk.tail(seq_len).copy()

# -----------------------------
# STEP 3. Create tf.data.Dataset with Shuffle and Batch
# -----------------------------
# First, determine the number of sensor features.
num_features = len(sensor_cols)

# Define the output signature using tf.TensorSpec.
output_signature = (
    tf.TensorSpec(shape=(seq_len, num_features), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.string)
)

# Create the dataset from the generator.
ds = tf.data.Dataset.from_generator(
    lambda: sequence_generator(csv_file, sensor_cols, target_col, seq_len, chunksize),
    output_signature=output_signature
)

# Shuffle the dataset to mix samples from different persons.
ds = ds.shuffle(buffer_size=shuffle_buffer_size)

# Batch the dataset.
ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Optionally, inspect a few batches to see unique labels.
unique_labels = set()
for X_batch, y_batch in ds.take(10):
    unique_labels.update(y_batch.numpy().tolist())
print("Unique labels in first 10 batches:", unique_labels)

# -----------------------------
# STEP 4. Build a Lookup Table for Encoding Labels
# -----------------------------
# If the initial unique labels are still just b'1', it might be that the first chunks are dominated by that label.
# We assume that over time, the dataset has all 25 unique labels.
# Here, we force the lookup table based on known possible labels (for example, '1' to '25'):
expected_labels = [str(i) for i in range(1, 26)]
print("Expected labels:", expected_labels)

keys_tensor = tf.constant(expected_labels)
vals_tensor = tf.constant([int(x) for x in expected_labels], dtype=tf.int32)
table = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor),
    default_value=-1
)

def encode_fn(X, y):
    return X, table.lookup(y)

ds = ds.map(encode_fn)

# -----------------------------
# STEP 5. Build the LSTM Model
# -----------------------------
input_shape = (seq_len, num_features)
num_classes = len(expected_labels)
model = Sequential([
    LSTM(64, input_shape=input_shape, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])
model.summary()

# -----------------------------
# STEP 6. Train the Model
# -----------------------------
# Note: Depending on the total number of batches (ds.cardinality()), you may want to set steps_per_epoch.
# For this example, we let model.fit iterate over the dataset.
history = model.fit(
    ds,
    epochs=10  # Adjust epochs as needed
)

# -----------------------------
# STEP 7. Save the Trained Model
# -----------------------------
model_save_path = "/content/drive/MyDrive/Major Project/behavioral_biometrics_model_lstm.h5"
model.save(model_save_path)
print("Model training complete and saved to:", model_save_path)


Sensor columns: ['Accelerometer X', 'Accelerometer Y', 'Accelerometer Z', 'Gyroscope X', 'Gyroscope Y', 'Gyroscope Z', 'Magnetometer X', 'Magnetometer Y', 'Magnetometer Z', 'Rotation Vector X', 'Rotation Vector Y', 'Rotation Vector Z', 'Tilt Detector X', 'Tilt Detector Y', 'Tilt Detector Z', 'Auto-rotation X', 'Auto-rotation Y', 'Auto-rotation Z', 'Motion X', 'Motion Y', 'Motion Z', 'Last Touch X', 'Last Touch Y']
Target column: Person
Unique labels in first 10 batches: {b'1'}
Expected labels: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25']


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 50, 64)              │          22,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 50, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 25)                  │             825 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 35,769 (139.72 KB)

 Trainable params: 35,769 (139.72 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
  15651/Unknown 5294s 335ms/step - accuracy: 0.9160 - loss: 0.2849

InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipyker